# 错误归因与根因分析

前置知识：本章 Notebook 2-4

本节目标：掌握归因三步法，学会从低分评估结果系统化定位问题根因，并制定修复方向。

## 一、归因三步法

拿到评估结果后，如果某些 case 的得分很低，不要急着改代码。先用以下三步定位根因：

### 第一步：看分层指标，确定哪一层出了问题
- 检索层指标低（Context Precision / Recall）→ 检索出了问题
- 生成层指标低（Faithfulness / Answer Relevancy）→ 生成出了问题
- 两层都低 → 可能是评估集本身的问题，也可能是系统性问题

### 第二步：层内细分，缩小范围
- 检索层：是 Recall 低（关键信息没检索到）还是 Precision 低（检索了太多无关内容）？
- 生成层：是 Faithfulness 低（模型在编故事）还是 Relevancy 低（答非所问）？

### 第三步：确认根因，制定行动
根据细分结果，找到具体的根因和修复方向。

## 二、归因矩阵

| 现象 | 细分 | 根因 | 修复方向 |
|------|------|------|----------|
| 检索层低分 | Recall 低 | 数据覆盖不够 | 补充文档、优化切分 |
| 检索层低分 | Recall 低 | Embedding 质量不足 | 换用更好的 embedding 模型 |
| 检索层低分 | Precision 低 | 噪声文档过多 | 调整 chunk_size、增加过滤 |
| 检索层低分 | Precision 低 | Query 表述与文档不匹配 | Query 改写、HyDE |
| 生成层低分 | Faithfulness 低 | LLM 幻觉 | 强化 prompt 约束、降低 temperature |
| 生成层低分 | Faithfulness 低 | Context 不充分，LLM 自行补充 | 增加检索 k 值、优化检索策略 |
| 生成层低分 | Relevancy 低 | Prompt 模板问题 | 优化 prompt 结构 |
| 生成层低分 | Relevancy 低 | Context 干扰，LLM 答偏了 | 添加 context 过滤、rerank |
| 两层都低 | — | 评估集质量问题 | 检查标准答案是否准确 |

In [ ]:
import os
import json
import re
import numpy as np
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

from modelscope import snapshot_download
model_dir = snapshot_download('BAAI/bge-small-zh-v1.5', cache_dir='./models')

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.chat_models import ChatZhipuAI
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

embedding = HuggingFaceEmbeddings(model_name=model_dir)
api_key = os.environ.get("ZHIPUAI_API_KEY")
llm = ChatZhipuAI(model="glm-4-flash", temperature=0.0, api_key=api_key)

pdf_path = "../3. 索引阶段/data/pumpkin_book.pdf"
persist_dir = "./chroma_db"

def clean_text(text: str) -> str:
    text = re.sub(r'→_→\n.*?←_←', '', text, flags=re.DOTALL)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def build_vectorstore(pdf_path, embedding, persist_directory="./chroma_db"):
    """构建或加载向量库，已有则复用"""
    if os.path.exists(persist_directory) and os.listdir(persist_directory):
        print(f"发现已存在的向量库: {persist_directory}，正在加载...")
        try:
            vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embedding)
            count = vectorstore._collection.count()
            print(f"✅ 加载成功！共 {count} 个文档块")
            return vectorstore
        except Exception as e:
            print(f"⚠️ 加载失败 ({e})，将重新构建...")
    print("开始构建向量库...")
    loader = PyMuPDFLoader(pdf_path)
    pdf_pages = loader.load()
    data_pages = pdf_pages[13:-13]
    for page in data_pages:
        page.page_content = clean_text(page.page_content)
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    splits = text_splitter.split_documents(data_pages)
    vectorstore = Chroma.from_documents(documents=splits, embedding=embedding, persist_directory=persist_directory)
    print(f"✅ 向量库构建完成并保存至 {persist_directory}，共 {len(splits)} 个文档块")
    return vectorstore

vectorstore = build_vectorstore(pdf_path, embedding, persist_dir)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_prompt = ChatPromptTemplate.from_template(
    "根据以下上下文回答问题。如果上下文中没有相关信息，请说'根据已有资料无法回答'。\n\n"
    "上下文：\n{context}\n\n问题：{question}\n\n回答："
)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)
print("RAG Pipeline 构建完成")

In [ ]:
import time
from tqdm import tqdm

eval_cases = [
    {"question": "什么是信息增益？", "ground_truth": "信息增益是指在得知某个特征的信息后，信息不确定性减少的程度。"},
    {"question": "为什么 SVM 要引入核函数？", "ground_truth": "当数据在原始空间线性不可分时，核函数将数据映射到高维空间使其线性可分。"},
    {"question": "请推荐几本量子计算的教材", "ground_truth": "根据已有资料无法回答。"},
    {"question": "L1 和 L2 正则化的区别？", "ground_truth": "L1 产生稀疏解，有特征选择效果；L2 使权重均匀缩小，防止权重过大。"},
    {"question": "2025年 GPT-5 的参数量是多少？", "ground_truth": "根据已有资料无法回答。"},
    {"question": "什么是交叉验证？", "ground_truth": "交叉验证将数据集分为k份，每次用k-1份训练、1份验证，重复k次取平均。"},
    {"question": "哪个算法最好？", "ground_truth": "没有绝对最好的算法，根据没有免费午餐定理。"},
    {"question": "决策树为什么容易过拟合？", "ground_truth": "决策树通过递归划分不断细分数据，容易学到训练数据中的噪声。"},
]

eval_results = []
for item in tqdm(eval_cases, desc="生成回答"):
    docs = retriever.invoke(item["question"])
    answer = rag_chain.invoke(item["question"])
    eval_results.append({
        **item,
        "answer": answer,
        "contexts": [doc.page_content for doc in docs],
    })
    time.sleep(0.5)

print(f"已生成 {len(eval_results)} 条回答")

In [ ]:
def multi_dim_judge(question, answer, ground_truth, context):
    judge_prompt = ChatPromptTemplate.from_template(
        "你是 RAG 评估专家。请从以下维度评分（0-10）：\n\n"
        "问题：{question}\n标准答案：{ground_truth}\n检索上下文：{context}\n系统回答：{answer}\n\n"
        "请按 JSON 格式返回：\n"
        '{{"context_recall": <上下文中是否包含回答所需的关键信息>, '
        '"context_precision": <检索结果中相关文档的占比>, '
        '"faithfulness": <回答是否基于上下文>, '
        '"answer_relevancy": <回答是否切题>}}'
    )
    chain = judge_prompt | llm | StrOutputParser()
    resp = chain.invoke({"question": question, "answer": answer, "ground_truth": ground_truth, "context": context[:2000]})
    try:
        resp = resp.strip()
        if resp.startswith("```"):
            resp = resp.split("\n", 1)[1].rsplit("```", 1)[0]
        scores = json.loads(resp)
        return {k: max(0, min(10, int(v))) for k, v in scores.items()}
    except:
        return {"context_recall": 5, "context_precision": 5, "faithfulness": 5, "answer_relevancy": 5}

for r in tqdm(eval_results, desc="多维评分"):
    ctx = "\n\n".join(r["contexts"][:3])
    r["scores"] = multi_dim_judge(r["question"], r["answer"], r["ground_truth"], ctx)
    time.sleep(1)

print("评分完成")

In [ ]:
for r in eval_results:
    r["avg_score"] = np.mean(list(r["scores"].values()))

sorted_results = sorted(eval_results, key=lambda x: x["avg_score"])

print("得分最低的 5 个 case：")
print("=" * 80)
for i, r in enumerate(sorted_results[:5]):
    print(f"\n--- Case {i+1} (均分: {r['avg_score']:.1f}) ---")
    print(f"问题：{r['question']}")
    print(f"回答：{r['answer'][:150]}...")
    print(f"评分：{r['scores']}")
    print(f"检索到的上下文片段数：{len(r['contexts'])}")
    print(f"上下文预览：{r['contexts'][0][:100]}...")

## 三、自动归因

根据分层指标的得分模式，自动判断问题根因。

In [ ]:
def diagnose(case):
    """根据分层指标自动归因"""
    s = case["scores"]

    diagnoses = []

    if s["context_recall"] < 5:
        diagnoses.append({
            "layer": "检索层",
            "issue": "召回不足",
            "detail": "关键信息未被检索到",
            "fix": "补充文档覆盖、优化 embedding 模型、增大检索 k 值"
        })

    if s["context_precision"] < 5:
        diagnoses.append({
            "layer": "检索层",
            "issue": "精度不足",
            "detail": "检索了太多无关文档",
            "fix": "减小 chunk_size、添加 rerank、优化 query"
        })

    if s["faithfulness"] < 6:
        diagnoses.append({
            "layer": "生成层",
            "issue": "幻觉",
            "detail": "LLM 生成了上下文中没有的内容",
            "fix": "强化 prompt 约束、降低 temperature、添加引用要求"
        })

    if s["answer_relevancy"] < 7:
        diagnoses.append({
            "layer": "生成层",
            "issue": "答非所问",
            "detail": "回答没有针对用户问题",
            "fix": "优化 prompt 模板、添加 query 重述"
        })

    if not diagnoses:
        diagnoses.append({
            "layer": "综合",
            "issue": "表现可接受",
            "detail": "各维度得分均在合理范围内",
            "fix": "无需紧急修复"
        })

    return diagnoses

print("归因分析报告")
print("=" * 80)
for i, r in enumerate(sorted_results[:5]):
    print(f"\n--- Case {i+1}: {r['question']} ---")
    print(f"评分：{r['scores']}")
    diags = diagnose(r)
    for d in diags:
        print(f"  [{d['layer']}] {d['issue']}：{d['detail']}")
        print(f"  修复方向：{d['fix']}")

In [ ]:
print(f"{'问题':<20} {'层级':<8} {'问题类型':<8} {'修复方向':<30}")
print("-" * 70)
for r in sorted_results[:5]:
    diags = diagnose(r)
    for d in diags:
        q_short = r["question"][:18]
        print(f"{q_short:<20} {d['layer']:<8} {d['issue']:<8} {d['fix'][:28]:<30}")

## 四、从归因到行动

归因分析给出了问题的根因和修复方向。下面是每种根因对应的具体修复措施：

### 检索层修复
| 根因 | 修复措施 | 预期效果 |
|------|---------|----------|
| 召回不足 | 增大 k 值 / 换用更强的 embedding | Context Recall 提升 |
| 召回不足 | 补充文档、优化切分策略 | 覆盖更多知识点 |
| 精度不足 | 添加 Rerank / Cross-Encoder | Context Precision 提升 |
| 精度不足 | 减小 chunk_size、增加 overlap | 减少噪声文档 |

### 生成层修复
| 根因 | 修复措施 | 预期效果 |
|------|---------|----------|
| 幻觉 | 在 prompt 中强化"仅基于上下文回答" | Faithfulness 提升 |
| 幻觉 | 降低 temperature | 减少创造性输出 |
| 答非所问 | 在 prompt 中重述用户问题 | Answer Relevancy 提升 |
| 答非所问 | 添加 query 改写 | 更好理解用户意图 |

### 修复优先级
1. **检索优先**：如果检索层得分低，先修检索，因为"检索不到就不可能回答好"
2. **一次改一个变量**：每次只修一个环节，跑评估看效果
3. **回归测试**：修完后跑完整评估集，确保没有引入新问题

## 五、小结

### 关键要点
1. **归因三步法**：分层 → 细分 → 定位根因，系统化而非猜测
2. **归因矩阵**：每种异常模式都有明确的根因和修复方向
3. **自动归因**：可以编写规则自动分类，加速问题定位
4. **检索优先**：检索层是基础，优先修复检索问题

### 实践建议
1. 每次低分时都走一遍归因流程，避免盲目改动
2. 记录归因历史，形成团队的问题知识库
3. 定期回顾归因报告，发现系统性问题

### 参考文献
- [Diagnosing RAG Pipeline Failures](https://docs.ragas.io/)